In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn streamlit

In [ ]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

In [ ]:
import kagglehub

path = kagglehub.dataset_download("emmarex/plantdisease")
print("Dataset downloaded to:", path)

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("emmarex/plantdisease")
print("Dataset downloaded to:", path)

In [ ]:
import os

dataset_path = "/kaggle/input/plantdisease"

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(indent + os.path.basename(root) + "/")

    if level >= 2:
        dirs[:] = []

    for file in files[:3]:
        print(indent + "    " + file)

In [ ]:
import os

dataset_path = "/kaggle/input/plantdisease"

total_images = 0
classes = []

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        images = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        if images:
            classes.append(folder)
            total_images += len(images)
            print(f"{folder}: {len(images)} images")

print("\nTotal classes:", len(classes))
print("Total images:", total_images)

In [ ]:
import os

dataset_path = "/kaggle/input/plantdisease"

for root, dirs, files in os.walk(dataset_path):
    print("Folder:", root)
    print("Subfolders:", dirs[:10])
    print("Number of files:", len(files))
    print("-" * 50)

    if root.count(os.sep) >= dataset_path.count(os.sep) + 3:
        dirs[:] = []

In [ ]:
dataset_path = "/kaggle/input/plantdisease/PlantVillage"

In [ ]:
import os

dataset_path = "/kaggle/input/plantdisease/PlantVillage"

total_images = 0
classes = []

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        images = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        if images:
            classes.append(folder)
            total_images += len(images)
            print(f"{folder}: {len(images)} images")

print("\nTotal classes:", len(classes))
print("Total images:", total_images)

In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image

plt.figure(figsize=(12, 8))

for i in range(6):
    folder = random.choice(classes)
    folder_path = os.path.join(dataset_path, folder)
    image_name = random.choice(os.listdir(folder_path))

    image = Image.open(os.path.join(folder_path, image_name))

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(folder.replace("_", " ")[:30])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names

print("Number of classes:", len(class_names))
print("Classes:", class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

val_dataset = val_dataset.prefetch(
    buffer_size=AUTOTUNE
)

print("Training and validation datasets are optimized!")

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(15, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Accuracy graph
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss graph
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_graphs.png')
plt.show()

In [ ]:
val_loss, val_accuracy = model.evaluate(val_dataset)

print("Final Validation Accuracy:", round(val_accuracy * 100, 2), "%")
print("Final Validation Loss:", round(val_loss, 4))

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in val_dataset:
    predictions = model(images, training=False)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions.numpy(), axis=1))

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

fig, ax = plt.subplots(figsize=(16, 14))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    xticks_rotation=90,
    cmap="Blues",
    values_format="d"
)

plt.title("Plant Disease Detection - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Plant Disease Detection - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig("/content/confusion_matrix.png", dpi=200)
plt.show()


In [ ]:
from google.colab import files
files.download("/content/confusion_matrix.png")

In [ ]:
import json

# Save trained CNN model
model.save("/content/plant_disease_model.keras")

# Save class names
with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)

# Save training graphs
print("Model saved successfully!")
print("Class names saved successfully!")

In [ ]:
from google.colab import files

files.download("/content/plant_disease_model.keras")
files.download("/content/class_names.json")
files.download("/content/training_graphs.png")

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
import tensorflow as tf
import json

model = tf.keras.models.load_model(
    "/content/plant_disease_model.keras",
    compile=False
)

with open("/content/class_names.json", "r") as f:
    class_names = json.load(f)

print("Model loaded successfully!")
print("Number of classes:", len(class_names))
print("Classes:", class_names)

In [ ]:
%%writefile /content/app.py

import streamlit as st
import tensorflow as tf
import numpy as np
import json
from PIL import Image, ImageOps

st.set_page_config(
    page_title="Plant Disease Detection",
    page_icon="🌿",
    layout="centered"
)

@st.cache_resource
def load_model():
    model = tf.keras.models.load_model(
        "/content/plant_disease_model.keras",
        compile=False
    )
    return model

@st.cache_data
def load_classes():
    with open("/content/class_names.json", "r") as f:
        return json.load(f)

model = load_model()
class_names = load_classes()

st.title("🌿 AI-Based Plant Disease Detection")
st.write("Upload a plant leaf image to identify its disease using CNN.")

uploaded_file = st.file_uploader(
    "Choose a leaf image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file)
    image = ImageOps.exif_transpose(image).convert("RGB")

    st.image(image, caption="Uploaded Leaf Image", use_container_width=True)

    if st.button("🔍 Detect Disease"):
        with st.spinner("Analyzing leaf image..."):

            resized_image = image.resize((128, 128))

            image_array = np.array(resized_image, dtype=np.float32)
            image_array = np.expand_dims(image_array, axis=0)

            predictions = model.predict(image_array, verbose=0)

            predicted_index = np.argmax(predictions[0])
            confidence = float(predictions[0][predicted_index]) * 100

            disease_name = class_names[predicted_index]

            display_name = disease_name.replace("_", " ").replace("__", " ")

            st.success("Analysis completed!")

            st.subheader("Prediction")
            st.write(f"**Detected Class:** {display_name}")
            st.write(f"**Confidence:** {confidence:.2f}%")

            st.progress(confidence / 100)

            st.subheader("Top 3 Predictions")

            top_indices = np.argsort(predictions[0])[-3:][::-1]

            for index in top_indices:
                name = class_names[index].replace("_", " ").replace("__", " ")
                score = float(predictions[0][index]) * 100

                st.write(f"{name}: {score:.2f}%")

st.divider()
st.caption(
    "Educational project using a CNN trained on the PlantVillage dataset. "
    "Predictions may differ from real-world plant conditions."
)

In [ ]:
import os
print(os.path.exists("/content/app.py"))

In [ ]:
!streamlit run /content/app.py &>/content/streamlit.log &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3MhANBQooQYuA1AXnv1ZwSd0FCO_7racBPisv4puvpzM6vAXf")

ngrok.kill()
public_url = ngrok.connect(8501)

print("Your application link:", public_url)

In [ ]:
%%writefile /content/app.py

import streamlit as st
import tensorflow as tf
import numpy as np
import json
from PIL import Image, ImageOps

st.set_page_config(
    page_title="Plant Disease Detection",
    page_icon="🌿",
    layout="wide"
)

@st.cache_resource
def load_model():
    return tf.keras.models.load_model(
        "/content/plant_disease_model.keras",
        compile=False
    )

@st.cache_data
def load_classes():
    with open("/content/class_names.json", "r") as f:
        return json.load(f)

model = load_model()
class_names = load_classes()

disease_info = {
    "Early_blight": {
        "description": "A fungal disease that affects leaves and may reduce plant growth.",
        "symptoms": "Brown circular spots with concentric rings and yellowing around lesions.",
        "cause": "Caused by the fungus Alternaria solani.",
        "prevention": "Remove infected leaves, avoid overhead watering and maintain good spacing."
    },
    "Late_blight": {
        "description": "A serious disease that can spread quickly in cool, wet conditions.",
        "symptoms": "Dark, water-soaked leaf patches that may develop pale growth underneath in humid conditions.",
        "cause": "Caused by Phytophthora infestans, a fungus-like organism.",
        "prevention": "Remove infected plant material, avoid wet foliage and use suitable disease management practices."
    },
    "Bacterial_spot": {
        "description": "A bacterial disease affecting pepper and tomato leaves.",
        "symptoms": "Small dark spots that may develop yellow margins.",
        "cause": "Caused by bacterial pathogens in the Xanthomonas group.",
        "prevention": "Use healthy seeds, avoid handling wet plants and remove infected debris."
    },
    "Leaf_Mold": {
        "description": "A fungal disease commonly found on tomato plants in humid conditions.",
        "symptoms": "Yellow patches on upper leaf surfaces and olive-green mold underneath.",
        "cause": "Caused by Passalora fulva.",
        "prevention": "Improve ventilation, reduce humidity and avoid overhead irrigation."
    },
    "Septoria_leaf_spot": {
        "description": "A fungal disease that commonly affects tomato leaves.",
        "symptoms": "Small circular spots with gray centers and dark edges.",
        "cause": "Caused by Septoria lycopersici.",
        "prevention": "Remove infected leaves, mulch soil and avoid splashing water onto leaves."
    },
    "Spider_mites": {
        "description": "Tiny pests that feed on plant cells.",
        "symptoms": "Fine yellow or pale speckles, sometimes with webbing on leaves.",
        "cause": "Caused by spider mite infestation.",
        "prevention": "Inspect leaf undersides and manage infestations using suitable integrated pest management."
    },
    "Target_Spot": {
        "description": "A fungal disease affecting tomato foliage.",
        "symptoms": "Brown circular lesions with target-like rings.",
        "cause": "Caused by Corynespora cassiicola.",
        "prevention": "Remove infected debris, improve airflow and avoid prolonged leaf wetness."
    },
    "YellowLeaf": {
        "description": "A viral disease that can seriously affect tomato growth.",
        "symptoms": "Yellowing, upward curling leaves and reduced plant growth.",
        "cause": "Tomato yellow leaf curl virus, commonly spread by whiteflies.",
        "prevention": "Control whiteflies, remove infected plants and use resistant varieties."
    },
    "mosaic_virus": {
        "description": "A viral disease that affects leaf development.",
        "symptoms": "Mosaic-like light and dark green patterns and distorted leaves.",
        "cause": "Tomato mosaic virus.",
        "prevention": "Use clean tools, healthy seeds and remove infected plants."
    },
    "healthy": {
        "description": "The model identifies this leaf as belonging to a healthy class.",
        "symptoms": "No characteristic disease symptoms were detected by the model.",
        "cause": "No disease indicated by this classification.",
        "prevention": "Continue proper watering, balanced nutrition and regular plant monitoring."
    }
}

def get_info(name):
    for key, value in disease_info.items():
        if key.lower() in name.lower():
            return value

    if "healthy" in name.lower():
        return disease_info["healthy"]

    return {
        "description": "The model identified a possible plant disease.",
        "symptoms": "Symptoms vary depending on the disease.",
        "cause": "A specific cause requires further confirmation.",
        "prevention": "Monitor the plant and seek local agricultural advice."
    }

st.title("🌿 AI-Based Plant Disease Detection")
st.write("An intelligent CNN-based system for identifying plant leaf diseases.")

st.info("Upload a clear plant leaf image to get an AI-based prediction.")

uploaded_file = st.file_uploader(
    "Upload Plant Leaf Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file:
    image = ImageOps.exif_transpose(
        Image.open(uploaded_file)
    ).convert("RGB")

    st.image(image, caption="Uploaded Leaf", width=350)

    if st.button("🔍 Detect Disease", type="primary"):

        with st.spinner("Analyzing image..."):

            resized = image.resize((128, 128))
            array = np.array(resized, dtype=np.float32)
            array = np.expand_dims(array, axis=0)

            predictions = model.predict(array, verbose=0)[0]

            index = int(np.argmax(predictions))
            confidence = float(predictions[index]) * 100

            disease = class_names[index]
            display_name = disease.replace("__", " ").replace("_", " ")

            info = get_info(disease)

        st.success("Analysis completed!")

        st.subheader("🔬 Prediction Result")

        col1, col2 = st.columns(2)

        with col1:
            st.metric("Detected Class", display_name)

        with col2:
            st.metric("Confidence", f"{confidence:.2f}%")

        st.progress(float(predictions[index]))

        st.subheader("📋 Disease Information")

        st.write("**Description:**", info["description"])
        st.write("**Symptoms:**", info["symptoms"])
        st.write("**Cause:**", info["cause"])
        st.write("**Prevention:**", info["prevention"])

        st.subheader("📊 Top 3 Predictions")

        top_indices = np.argsort(predictions)[-3:][::-1]

        for i in top_indices:
            name = class_names[i].replace("__", " ").replace("_", " ")
            score = float(predictions[i]) * 100
            st.write(f"**{name}:** {score:.2f}%")

        st.warning(
            "This is an educational AI prediction, not a confirmed "
            "agricultural diagnosis. Verify suspected diseases before treatment."
        )

st.divider()
st.caption("Developed using Python, TensorFlow, CNN and Streamlit.")

In [ ]:
!pkill -f "streamlit run" || true

In [ ]:
!streamlit run /content/app.py &>/content/streamlit.log &

In [ ]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8501)

print("Website link:", public_url)